In [ ]:
# # PURPOSE OF NOTEBOOK

# Gold layer should mainly focus on:
# aggregations
# KPIs
# analytical tables
# business metrics
# dimensional models

# Imports

In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
col,sum,avg,count,countDistinct,when,lit,round,concat,concat_ws,row_number,dense_rank,rank,desc,asc,max,min,stddev,
variance,current_date,months_between,floor,trim,initcap,isnan,coalesce)

from pyspark.sql.types import (
StructType,StructField,StringType,IntegerType,BooleanType,DateType,DecimalType,DoubleType)

from pyspark.sql.window import Window

import warnings
warnings.filterwarnings("ignore")



# Spark Session

In [2]:

try:
    spark.stop()
except:
    pass
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("S3App")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 07:40:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 07:40:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/26 07:40:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/26 07:40:12 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


# spark

In [3]:
spark


# PATHS ,0.0 SILVER LAYER PARQUET AND GOLD LAYER TO PARQUET 
# ==================================================

# # 8.3 Cache Silver Dataset
#  WHY HERE?

 Because:

 joins expensive
 reused multiple times
 validations + Gold aggregations reuse it

 Avoid recomputation.



In [4]:
# ==============================
# 0.0 SILVER LAYER PARQUET PATHS
# ==============================
# Base path for Silver layer
SILVER_PATH = "/opt/spark-data/3_silver"

# Final Silver DataFrame
final_silver_df_file_path_to_parquet = f"{SILVER_PATH}/final_silver_parquet"

# Silver parquet paths
ball_by_ball_file_path_to_parquet = f"{SILVER_PATH}/ball_by_ball_parquet"
match_file_path_to_parquet = f"{SILVER_PATH}/match_parquet"
player_file_path_to_parquet = f"{SILVER_PATH}/player_parquet"
player_match_file_path_to_parquet = f"{SILVER_PATH}/player_match_parquet"
team_file_path_to_parquet = f"{SILVER_PATH}/team_parquet"



# ============================================
# 0.0 GOLD LAYER PATHS
# ============================================

GOLD_PATH = "/opt/spark-data/4_gold"

BATTING_SUMMARY_PATH = f"{GOLD_PATH}/batting_summary"
BOWLING_SUMMARY_PATH = f"{GOLD_PATH}/bowling_summary"
DEATH_OVER_ANALYSIS_PATH = f"{GOLD_PATH}/death_over_analysis"
MATCH_FACT_TABLE_PATH = f"{GOLD_PATH}/match_fact_table"
ORANGE_CAP_PATH = f"{GOLD_PATH}/orange_cap"
TEAM_PERFORMANCE_PATH = f"{GOLD_PATH}/team_performance"
VENUE_ANALYSIS_PATH = f"{GOLD_PATH}/venue_analysis"

 # AND READING FILES

In [5]:

# ==============================
# 0.1 READ SILVER DATA
# ==============================

final_silver_df = spark.read.parquet(
    final_silver_df_file_path_to_parquet
)

ball_by_ball_df = spark.read.parquet(
    ball_by_ball_file_path_to_parquet
)

match_df = spark.read.parquet(
    match_file_path_to_parquet
)

player_df = spark.read.parquet(
    player_file_path_to_parquet
)

player_match_df = spark.read.parquet(
    player_match_file_path_to_parquet
)

team_df = spark.read.parquet(
    team_file_path_to_parquet
)

In [6]:
# 0.2 Verify Data
# final_silver_df.printSchema()

# final_silver_df.show(2)

In [7]:
print(final_silver_df.columns)
print(len(final_silver_df.columns))
print(player_df.columns)

['match_id', 'over_id', 'ball_id', 'innings_no', 'team_batting', 'team_bowling', 'striker_batting_position', 'extra_type', 'runs_scored', 'extra_runs', 'wides', 'legbyes', 'byes', 'noballs', 'penalty', 'bowler_extras', 'out_type', 'caught', 'bowled', 'run_out', 'lbw', 'retired_hurt', 'stumped', 'caught_and_bowled', 'hit_wicket', 'obstructingfeild', 'bowler_wicket', 'match_date', 'season', 'striker', 'non_striker', 'bowler', 'player_out', 'fielders', 'striker_match_sk', 'strikersk', 'nonstriker_match_sk', 'nonstriker_sk', 'fielder_match_sk', 'fielder_sk', 'bowler_match_sk', 'bowler_sk', 'playerout_match_sk', 'battingteam_sk', 'bowlingteam_sk', 'keeper_catch', 'player_out_sk', 'matchdatesk', 'total_runs', 'is_four', 'is_six', 'is_dot_ball', 'is_wicket', 'over_ball', 'innings_phase', 'strike_rotation', 'match_sk', 'team1', 'team2', 'season_year', 'venue_name', 'city_name', 'country_name', 'toss_winner', 'match_winner', 'toss_name', 'win_type', 'outcome_type', 'manofmach', 'win_margin', 'c

In [8]:
player_df.select('player_id', 'player_name').show(2)

+---------+-----------+
|player_id|player_name|
+---------+-----------+
|      152|   A Mukund|
|      444|  Mb Parmar|
+---------+-----------+
only showing top 2 rows



# checking AQE status is its true bydefault

In [9]:

# checking AQE status is its true bydefault

spark.conf.get("spark.sql.adaptive.enabled")

# Disable AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.get("spark.sql.adaptive.enabled")



'false'

In [10]:
#0.3 
print(final_silver_df.columns)
final_silver_df.show(2)

['match_id', 'over_id', 'ball_id', 'innings_no', 'team_batting', 'team_bowling', 'striker_batting_position', 'extra_type', 'runs_scored', 'extra_runs', 'wides', 'legbyes', 'byes', 'noballs', 'penalty', 'bowler_extras', 'out_type', 'caught', 'bowled', 'run_out', 'lbw', 'retired_hurt', 'stumped', 'caught_and_bowled', 'hit_wicket', 'obstructingfeild', 'bowler_wicket', 'match_date', 'season', 'striker', 'non_striker', 'bowler', 'player_out', 'fielders', 'striker_match_sk', 'strikersk', 'nonstriker_match_sk', 'nonstriker_sk', 'fielder_match_sk', 'fielder_sk', 'bowler_match_sk', 'bowler_sk', 'playerout_match_sk', 'battingteam_sk', 'bowlingteam_sk', 'keeper_catch', 'player_out_sk', 'matchdatesk', 'total_runs', 'is_four', 'is_six', 'is_dot_ball', 'is_wicket', 'over_ball', 'innings_phase', 'strike_rotation', 'match_sk', 'team1', 'team2', 'season_year', 'venue_name', 'city_name', 'country_name', 'toss_winner', 'match_winner', 'toss_name', 'win_type', 'outcome_type', 'manofmach', 'win_margin', 'c

26/06/26 07:40:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-------+-------+----------+------------+------------+------------------------+----------+-----------+----------+-----+-------+----+-------+-------+-------------+--------------+------+------+-------+-----+------------+-------+-----------------+----------+----------------+-------------+----------+------+-------+-----------+------+----------+--------+----------------+---------+-------------------+-------------+----------------+----------+---------------+---------+------------------+--------------+--------------+------------+-------------+-----------+----------+-------+------+-----------+---------+---------+-------------+---------------+--------+--------------------+----------------+-----------+--------------------+---------+------------+--------------------+-------------------+---------+--------+------------+---------+----------+----------+--------------------+-----------------------------+------------+--------------+----------------+---------+-----------+
|match_id|over_id|ball

# ============================================
# 1.0 MATCH FACT TABLE
# ============================================

In [11]:

# ============================================
# 1.0 MATCH FACT TABLE
# ============================================

match_fact_df = (
    final_silver_df
    .groupBy(
        "match_id",
        "season_year",
        "venue_name",
        "team_batting",
        "team_bowling"
    )
    .agg(
        sum("runs_scored").alias("total_runs"),

        sum("extra_runs").alias("extras"),

        count("*").alias("balls"),

        sum(
            when(col("is_wicket") == 1, 1)
            .otherwise(0)
        ).alias("wickets")
    )
)




In [12]:

# dataset is small enough that Spark optimized the aggregation into a single partition output.
print(match_fact_df.rdd.getNumPartitions())
# match_fact_df = match_fact_df.coalesce(2)

# print(match_fact_df.rdd.getNumPartitions())


200


In [13]:
# 0.4 validations

print(match_fact_df.columns)

print(len(match_fact_df.columns))

['match_id', 'season_year', 'venue_name', 'team_batting', 'team_bowling', 'total_runs', 'extras', 'balls', 'wickets']
9


In [14]:
# # ============================================
# 1.1 SHOW OUTPUT
# ============================================


# match_fact_df.show(5)

In [15]:
# ============================================
# 1.2 OPTIMIZATION coalesce Prevents small file problem
# Optimized parquet output
# Less metadata overhead
# slower reads
# metadata overhead
# poor cloud performance
# expensive object storage operations
# slower BI queries

# ============================================
# match_fact_df
#     .coalesce(2)
#     .write
#     .mode("overwrite")
#     .parquet("/opt/spark-data/ipl_data/4_gold/match_fact_table")
# or
# ============================================



print(match_fact_df.rdd.getNumPartitions())

# match_fact_df = match_fact_df.coalesce(2)   #"AQE already optimized small datasets to a single partition, so additional coalesce operations were unnecessary."

print(match_fact_df.rdd.getNumPartitions())

# Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("Enable AQE")


# match_fact_df = match_fact_df.coalesce(2)


200
200
Enable AQE


In [16]:
print(match_fact_df.rdd.getNumPartitions())

# ============================================
# 1.3 WRITE GOLD TABLE
# ============================================


match_fact_df\
.write\
.mode("overwrite")\
.parquet(MATCH_FACT_TABLE_PATH)



200


# ============================================
# 2.0 BATTING SUMMARY
# ============================================


In [17]:


# Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

# ============================================
# 2.0 BATTING SUMMARY
# ============================================

batting_summary_df = (
    final_silver_df
    .groupBy(
        "striker",
        "striker_name"
    )
    .agg(
        sum("runs_scored").alias("total_runs"),

        count("*").alias("balls_faced"),

        sum(
            when(col("is_four") == 1, 1)
            .otherwise(0)
        ).alias("fours"),

        sum(
            when(col("is_six") == 1, 1)
            .otherwise(0)
        ).alias("sixes"),

        countDistinct("match_id").alias("matches")
    )
)

# ============================================
# STRIKE RATE
# ============================================

batting_summary_df = batting_summary_df.withColumn(
    "strike_rate",

    round(
        (col("total_runs") / col("balls_faced")) * 100,
        2
    )
)



In [18]:
# 0.4 validations

print(batting_summary_df.columns)

print(len(batting_summary_df.columns))

['striker', 'striker_name', 'total_runs', 'balls_faced', 'fours', 'sixes', 'matches', 'strike_rate']
8


In [19]:
# ============================================
# 2.1 SHOW OUTPUT
# ============================================

batting_summary_df.show(5)


[Stage 11:=====================>                                    (3 + 5) / 8]

+-------+--------------+----------+-----------+-----+-----+-------+-----------+
|striker|  striker_name|total_runs|balls_faced|fours|sixes|matches|strike_rate|
+-------+--------------+----------+-----------+-----+-----+-------+-----------+
|    395| Anureet Singh|        33|         43|    2|    1|      7|      76.74|
|    141|   Younis Khan|         3|          7|    0|    0|      1|      42.86|
|    427|       Hm Amla|       577|        418|   60|   21|     16|     138.04|
|    307|    Mn Samuels|       161|        174|    9|    7|     14|      92.53|
|    403|Mj Mcclenaghan|        63|         45|    2|    6|     12|      140.0|
+-------+--------------+----------+-----------+-----+-----+-------+-----------+
only showing top 5 rows



In [20]:
# ============================================
# 2.2 OPTIONAL OPTIMIZATION
# ============================================

# batting_summary_df = batting_summary_df.coalesce(2)


# ============================================
# 2.3 WRITE GOLD TABLE
# ============================================


print(batting_summary_df.rdd.getNumPartitions())


batting_summary_df.write.mode("overwrite").parquet(BATTING_SUMMARY_PATH)

1


# ============================================
# 3.0 BOWLING SUMMARY
# ============================================

In [21]:
final_silver_df.columns
final_silver_df.select()

DataFrame[]

In [22]:
# ============================================
# 3.0 BOWLING SUMMARY
# ============================================
# striker_name column is player_name


bowling_summary_df = (
    final_silver_df
    .groupBy("bowler", "bowler_name")
    .agg(
        sum("total_runs").alias("runs_conceded"),

        count("*").alias("balls_bowled"),

        sum(
            when(col("is_wicket") == 1, 1)
            .otherwise(0)
        ).alias("wickets")
    )
)

# ============================================
# OVERS CALCULATION
# ============================================

bowling_summary_df = bowling_summary_df.withColumn(
    "overs",

    round(
        col("balls_bowled") / 6,
        2
    )
)

# ============================================
# ECONOMY CALCULATION
# ============================================

bowling_summary_df = bowling_summary_df.withColumn(
    "economy",

    round(
        col("runs_conceded") / col("overs"),
        2
    )
)




In [23]:
# 3.1 validations

print(bowling_summary_df.columns)

print(len(bowling_summary_df.columns))

['bowler', 'bowler_name', 'runs_conceded', 'balls_bowled', 'wickets', 'overs', 'economy']
7


In [24]:

# ============================================
# 3.2 SHOW OUTPUT
# ============================================

bowling_summary_df.show(5)


+------+--------------+-------------+------------+-------+------+-------+
|bowler|   bowler_name|runs_conceded|balls_bowled|wickets| overs|economy|
+------+--------------+-------------+------------+-------+------+-------+
|   395| Anureet Singh|          594|         406|     19| 67.67|   8.78|
|   403|Mj Mcclenaghan|         1356|         982|     56|163.67|   8.28|
|   307|    Mn Samuels|          285|         219|     10|  36.5|   7.81|
|   325|A Ashish Reddy|          400|         270|     19|  45.0|   8.89|
|   256| Harmeet Singh|          747|         574|     31| 95.67|   7.81|
+------+--------------+-------------+------------+-------+------+-------+
only showing top 5 rows



In [25]:
# ============================================
# 3.3 OPTIONAL OPTIMIZATION
# ============================================

# bowling_summary_df = bowling_summary_df.coalesce(2)

# ============================================
# WRITE GOLD TABLE
# ============================================

bowling_summary_df.write.mode("overwrite").parquet(BOWLING_SUMMARY_PATH)


# 4. Window analysis 
# ============================================
# ORANGE CAP ANALYSIS
# ============================================

In [26]:
# ============================================
# 4.0 ORANGE CAP ANALYSIS
# ============================================

orange_cap_df = (
    final_silver_df
    .groupBy(
        "season_year",
        "striker",
        "striker_name"
    )
    .agg(
        sum("runs_scored").alias("season_runs")
    )
)

# ============================================
# WINDOW SPECIFICATION
# ============================================

window_spec = (
    Window
    .partitionBy("season_year")
    .orderBy(desc("season_runs"))
)

# ============================================
# DENSE RANK
# ============================================

orange_cap_df = orange_cap_df.withColumn(
    "rank",

    dense_rank().over(window_spec)
)

# ============================================
# TOP RUN SCORER PER SEASON
# ============================================

orange_cap_df = orange_cap_df.filter(
    col("rank") == 1
)



In [27]:
# ============================================
# 4.1 SHOW OUTPUT validations
# ============================================

print(orange_cap_df.columns)

print(len(orange_cap_df.columns))

orange_cap_df.show(5)



['season_year', 'striker', 'striker_name', 'season_runs', 'rank']
5
+-----------+-------+------------+-----------+----+
|season_year|striker|striker_name|season_runs|rank|
+-----------+-------+------------+-----------+----+
|       2008|    100|    Se Marsh|        616|   1|
|       2009|     18|   Ml Hayden|        572|   1|
|       2010|    133|Sr Tendulkar|        617|   1|
|       2011|    162|    Ch Gayle|        604|   1|
|       2012|    162|    Ch Gayle|        733|   1|
+-----------+-------+------------+-----------+----+
only showing top 5 rows



In [28]:
# ============================================
# 4.3 OPTIONAL OPTIMIZATION
# ============================================

# orange_cap_df = orange_cap_df.coalesce(1)

# ============================================
# WRITE GOLD TABLE
# ============================================

orange_cap_df.write.mode("overwrite").parquet(ORANGE_CAP_PATH)

# ============================================
# 5.0 DEATH OVER ANALYSIS
# ============================================

In [29]:
# ============================================
# 5.0 DEATH OVER ANALYSIS
# ============================================

death_over_analysis_df = (
    final_silver_df

    .filter(col("over_id") >= 16)

    .groupBy("team_batting")

    .agg(
        sum("total_runs").alias("death_runs"),

        count("*").alias("balls"),

        sum("is_four").alias("fours"),

        sum("is_six").alias("sixes"),

        sum("is_wicket").alias("wickets_lost")
    )
)

# ============================================
# DEATH OVER STRIKE RATE
# ============================================

death_over_analysis_df = death_over_analysis_df.withColumn(
    "death_strike_rate",

    round(
        (col("death_runs") / col("balls")) * 100,
        2
    )
)

# ============================================
# RUN RATE
# ============================================

death_over_analysis_df = death_over_analysis_df.withColumn(
    "run_rate",

    round(
        (col("death_runs") / col("balls")) * 6,
        2
    )
)



In [30]:
# ============================================
# 5.1 SHOW OUTPUT VALIDATIONS
# ============================================

print(death_over_analysis_df.columns)

print(len(death_over_analysis_df.columns))

death_over_analysis_df.show(5)


['team_batting', 'death_runs', 'balls', 'fours', 'sixes', 'wickets_lost', 'death_strike_rate', 'run_rate']
8
+-------------------+----------+-----+-----+-----+------------+-----------------+--------+
|       team_batting|death_runs|balls|fours|sixes|wickets_lost|death_strike_rate|run_rate|
+-------------------+----------+-----+-----+-----+------------+-----------------+--------+
|                  7|      6328| 3925|  484|  325|         331|           161.22|    9.67|
|                 11|      2620| 1720|  169|  128|         145|           152.33|    9.14|
|Sunrisers Hyderabad|       618|  386|   57|   19|          27|            160.1|    9.61|
|                  3|      5946| 3682|  421|  277|         261|           161.49|    9.69|
|                  8|      3133| 2103|  217|  138|         198|           148.98|    8.94|
+-------------------+----------+-----+-----+-----+------------+-----------------+--------+
only showing top 5 rows



In [31]:
# ============================================
# 5.2 OPTIONAL OPTIMIZATION
# ============================================

# death_over_analysis_df = death_over_analysis_df.coalesce(1)


# ============================================
# WRITE GOLD TABLE
# ============================================

death_over_analysis_df.write.mode("overwrite").parquet(DEATH_OVER_ANALYSIS_PATH)

# ============================================
# 6.0 VENUE ANALYSIS
# ============================================

In [32]:
# ============================================
# 6.0 VENUE ANALYSIS
# ============================================

venue_analysis_df = (
    final_silver_df

    .groupBy("venue_name")

    .agg(
        round(
            avg("total_runs"),
            2
        ).alias("avg_runs_per_ball"),

        sum("total_runs").alias("total_runs_scored"),

        sum("is_wicket").alias("total_wickets"),

        sum("is_four").alias("total_fours"),

        sum("is_six").alias("total_sixes"),

        countDistinct("match_id").alias("matches")
    )
)

# ============================================
# AVERAGE RUNS PER MATCH
# ============================================

venue_analysis_df = venue_analysis_df.withColumn(
    "avg_runs_per_match",

    round(
        col("total_runs_scored") / col("matches"),
        2
    )
)

# ============================================
# BOUNDARY PERCENTAGE
# ============================================

venue_analysis_df = venue_analysis_df.withColumn(
    "boundary_percentage",

    round(
        (
            (
                (col("total_fours") * 4)
                +
                (col("total_sixes") * 6)
            )
            /
            col("total_runs_scored")
        ) * 100,
        2
    )
)



In [33]:
# ============================================
# 6.1 SHOW OUTPUT VALIDATIONS
# ============================================

print(venue_analysis_df.columns)

print(len(venue_analysis_df.columns))

venue_analysis_df.show(5)

['venue_name', 'avg_runs_per_ball', 'total_runs_scored', 'total_wickets', 'total_fours', 'total_sixes', 'matches', 'avg_runs_per_match', 'boundary_percentage']
9
+--------------------+-----------------+-----------------+-------------+-----------+-----------+-------+------------------+-------------------+
|          venue_name|avg_runs_per_ball|total_runs_scored|total_wickets|total_fours|total_sixes|matches|avg_runs_per_match|boundary_percentage|
+--------------------+-----------------+-----------------+-------------+-----------+-----------+-------+------------------+-------------------+
|Dubai Internation...|             1.21|             2064|           79|        167|         58|      7|            294.86|              49.22|
|Himachal Pradesh ...|             1.34|             2893|          104|        298|        108|      9|            321.44|               63.6|
|Sardar Patel Stad...|              1.3|             3767|          145|        306|        130|     12|            31

In [34]:
# ============================================
# 6.2 OPTIONAL OPTIMIZATION
# ============================================

# venue_analysis_df = venue_analysis_df.coalesce(1)



# ============================================
# WRITE GOLD TABLE
# ============================================


venue_analysis_df.write.mode("overwrite").parquet(VENUE_ANALYSIS_PATH)


# ============================================
# # 7.TEAM PERFORMANCE
# ============================================

In [35]:
# ============================================
# 7.0 TEAM PERFORMANCE ANALYSIS
# ============================================




team_performance_df = (
    final_silver_df
    .filter(col("match_winner") != "Tied")  # removing /filter tied matches
    .groupBy("match_winner")

    .agg(
        countDistinct("match_id").alias("wins"),

        sum("total_runs").alias("total_team_runs"),

        sum("is_wicket").alias("total_wickets_lost"),

        count("*").alias("balls_played"),

        sum("is_four").alias("total_fours"),

        sum("is_six").alias("total_sixes")
    )
)

# ============================================
# TEAM STRIKE RATE
# ============================================

team_performance_df = team_performance_df.withColumn(
    "team_strike_rate",

    round(
        (col("total_team_runs") / col("balls_played")) * 100,
        2
    )
)

# ============================================
# TEAM RUN RATE
# ============================================

team_performance_df = team_performance_df.withColumn(
    "run_rate",

    round(
        (col("total_team_runs") / col("balls_played")) * 6,
        2
    )
)

# ============================================
# BOUNDARY RUN PERCENTAGE
# ============================================

team_performance_df = team_performance_df.withColumn(
    "boundary_percentage",

    round(
        (
            (
                (col("total_fours") * 4)
                +
                (col("total_sixes") * 6)
            )
            /
            col("total_team_runs")
        ) * 100,
        2
    )
)


In [36]:
# ============================================
# 7.1 SHOW OUTPUT VALIDATIONS
# ============================================

print(team_performance_df.columns)

print(len(team_performance_df.columns))

team_performance_df.show(5)


['match_winner', 'wins', 'total_team_runs', 'total_wickets_lost', 'balls_played', 'total_fours', 'total_sixes', 'team_strike_rate', 'run_rate', 'boundary_percentage']
10
+--------------------+----+---------------+------------------+------------+-----------+-----------+----------------+--------+-------------------+
|        match_winner|wins|total_team_runs|total_wickets_lost|balls_played|total_fours|total_sixes|team_strike_rate|run_rate|boundary_percentage|
+--------------------+----+---------------+------------------+------------+-----------+-----------+----------------+--------+-------------------+
| Sunrisers Hyderabad|  42|          12671|               505|        9964|       1091|        395|          127.17|    7.63|              53.14|
| Chennai Super Kings|  79|          24696|               925|       18994|       2075|        860|          130.02|     7.8|               54.5|
|     Deccan Chargers|  29|           8739|               369|        7013|        710|        315| 

In [37]:

# ============================================
# 7.2 OPTIONAL OPTIMIZATION
# ============================================

# team_performance_df = team_performance_df.coalesce(1)


# ============================================
# WRITE GOLD TABLE
# ============================================


team_performance_df.write.mode("overwrite").parquet(TEAM_PERFORMANCE_PATH)

# Successfully generated Gold Layer analytics tables 
# in parquet format for downstream reporting and analysis:

In [38]:
# Successfully generated Gold Layer analytics tables 
# in parquet format for downstream reporting and analysis:

# batting_summary/
# bowling_summary/
# death_over_analysis/
# match_fact_table/
# orange_cap/
# team_performance/
# venue_analysis/

In [39]:
ls


01_ipl_data_ingestion.ipynb
02_ipl_data_validate_write_bronze.ipynb*
03_ipl_data_Transform_silver.ipynb*
04_ipl_data_Transformation_gold.ipynb*
05_spark_sql_analytic.ipynb
